# K5's between-video baseline: micro under per-video transforms (DoTA-CAP-dev, DADA-2000)

Rule: `core/docs/final/KIM_RUN.md` §21, written before any number. **PRELIMINARY (n = 1)**. Reads the `clip_scores.npz` files K5 s2024 (and LaGoVAD's public checkpoint, printed beside) already left on Drive: **nothing trains, nothing is extracted, no GPU.** Macro cannot move under any row (the tool raises if it does). Every row except the two oracles needs the whole video: a **protocol change, printed only**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight (score files and recorded read-outs) | no | 2-4 min |
| 4 | pytest on the touched code, **raises if red** | no | 1-3 min |
| 5 | `core.tools.k5_baseline_micro` -> **paste `REPORTS/k5_baseline_micro.md` to Claude** | no | 5-25 min (2,000 bootstrap draws x 8 AUCs per set; the frame counts are not known here, DoTA is the slow one) |

Outputs go to `Thesis-Final/outputs/kim_baseline/REPORTS`; Thesis and Thesis-V2 are not touched.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_F = '/content/drive/MyDrive/Thesis-Final'   # code, K5's reads, LaGoVAD's reads, every output
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = '/content/drive/MyDrive/Thesis/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = '/content/drive/MyDrive/Thesis/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/k5_baseline_micro.py').is_file(), 'no k5_baseline_micro -- re-upload kat-vad/'
assert '## 21. K5' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no section 21 -- re-upload'
assert 'groups: dict' in (REPO / 'core/tools/tad_clip_gate.py').read_text(), 'old tad_clip_gate -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402
from core.tools import protocol_b_eval, dada_whole_eval  # noqa: E402
from core.tools.zs_lagovad_readout import PROTOCOL_B  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, LAG = constants.KIM_MAIN_ARM, constants.ZS_LAGOVAD_ARM
KIM = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
ZS  = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME
REPORTS = constants.OUTPUT_ROOT / constants.BASEMICRO_OUTPUT_DIRNAME / 'REPORTS'
LOCAL = Path('/content/k5_baseline')
LOCAL.mkdir(parents=True, exist_ok=True)

# (label, dir with clip_scores.npz + its recorded read-out, read-out file, pooling of its recorded micro, bootstrap unit, required)
SETS = [
    ('DoTA-CAP-dev/K5',      KIM / 'eval' / 'dota' / K5,                protocol_b_eval.READOUT_JSON, 'minmax', 'dota', True),
    ('DADA-2000/K5',         KIM / 'eval' / 'dada' / K5,                dada_whole_eval.READOUT_JSON, 'raw',    'item', True),
    ('DoTA-CAP-dev/LaGoVAD', ZS / 'eval' / 'dota' / LAG / PROTOCOL_B,   protocol_b_eval.READOUT_JSON, 'minmax', 'dota', False),
    ('DADA-2000/LaGoVAD',    ZS / 'eval' / 'dada' / LAG,                dada_whole_eval.READOUT_JSON, 'raw',    'item', False),
]
use, missing = [], []
for label, d, readout, as_read, cluster, required in SETS:
    npz = d / protocol_b_eval.CLIP_SCORES_NPZ
    absent = [str(p) for p in (npz, d / readout) if not p.is_file()]
    if absent and required:
        missing += absent
    elif absent:
        print(f'skipping {label} (printed-beside row, files absent): {absent}')
    else:
        use.append((label, npz, as_read, cluster, d / readout))
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(use)} sets to read | outputs -> {REPORTS}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn scipy
python -c "import av, einops, faiss, sklearn, scipy; print('deps OK')"

## 4. Tests on the touched code (CPU), **raises if anything fails: stop and paste the output**

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_k5_baseline_micro.py',
     'core/tests/test_k5_clip_gate.py', 'core/tests/test_tad_clip_gate.py'])

## 5. The table -> **paste `REPORTS/k5_baseline_micro.md` to Claude**

Gates G1 (recorded macro / micro reproduce) and G2 (macro unchanged by every row) raise before any table is written. Rows and sentences: `KIM_RUN.md` §21 BM2-BM6.

In [ ]:
local = LOCAL / 'report'
cmd = [sys.executable, '-m', 'core.tools.k5_baseline_micro', '--out-dir', local]
for label, npz, as_read, cluster, readout in use:
    cmd += ['--set', label, npz, as_read, cluster, readout]
run(cmd)
publish(local, REPORTS)
print((REPORTS / f'{constants.BASEMICRO_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_baseline_micro.md to Claude')